<a href="https://colab.research.google.com/github/aahmad97/COMP324Project/blob/main/02_ml_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PCA

---
Loyola University Chicago  
COMP 379-001/479-001, Spring 2025, Machine Learning  
Instructor: Daniel Moreira (dmoreira1@luc.edu)  
More at https://danielmoreira.github.io/teaching/ml_spr25/

---

Practical examples and exercises about Principal Component Analysis (PCA)

Language: Python 3  

Needed libraries:
* NumPy (https://numpy.org/)
* Pandas (https://pandas.pydata.org/)
* Scikit-learn (https://scikit-learn.org/)
* matplotlib (https://matplotlib.org/)


## 0 - Toy-case Data Loading

In [ ]:
# downloads the toycase dataset for learning PCA
!pip install gdown==v5.1.0
!gdown 1FKmsRnz-cXARhFlRSJYSeulBMSjnRKVm

In [ ]:
# imported libraries
import numpy as np
print('NumPy version', np.__version__)

import pandas as pd
print('Pandas version', pd.__version__)

import matplotlib
print('Matplotlib version', matplotlib.__version__)

import sklearn
print('SciKit version', sklearn.__version__)

In [ ]:
# loads the toy-case dataset into memory
df_x = pd.read_csv('/content/toycase.csv', header=None)

# prints info
print('Data shape:', df_x.shape)

# first 5 samples
df_x.head(5)

In [ ]:
# data in numpy array format
X = df_x.values
print('Data shape:', X.shape)

In [ ]:
# plots the data
import matplotlib.pyplot as plt
plt.scatter(X[:, 0], X[:, 1])
plt.show()

## 1 - Data Standardization

In [ ]:
# numpy implementation
norm_X = (X - X.mean(axis = 0)) / X.std(axis = 0)
plt.scatter(norm_X[:, 0], norm_X[:, 1])
plt.show()

In [ ]:
# same thing, but with scikit scaler object
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# norm data
norm_X = scaler.fit_transform(X)

# data plot
plt.scatter(norm_X[:, 0], norm_X[:, 1])
plt.show()

## 2 - Covariance Matrix Computation

In [ ]:
# numpy implementation
cov_mat = norm_X.transpose() @ norm_X / (norm_X.shape[0] - 1)
cov_mat

In [ ]:
# same thing, but with numpy built-in function
cov_mat = np.cov(norm_X, rowvar=False) # one sample per row
cov_mat

## 3 - Eigenvectors and Eigenvalues Computation

In [ ]:
# numpy built-in function to compute eigenstuff
eigen_vals, eigen_vecs = np.linalg.eig(cov_mat)
print('Eigenvalues:', eigen_vals)
print('Eigenvectors:\n', eigen_vecs)

## 4 - Eigenvalues Selection and Projection Matrix Preparation

### Variance Ratio Analysis

In [ ]:
import matplotlib.pyplot as plt

# computes the variance rations
eigen_sum = sum(eigen_vals)
var_dist = [i / eigen_sum for i in sorted(eigen_vals, reverse=True)] # from the largest to the smallest
plt.bar(range(len(eigen_vals)), var_dist, alpha=0.5, align='center', label='Variance Ratios')

# cumulative variances
cum_var = np.cumsum(var_dist)
plt.step(range(len(eigen_vals)), cum_var, where='mid', label='Cumulative Variance')

# graph plot setup
plt.ylabel('Variance Ratio')
plt.ylabel('Principal Components')
plt.legend(loc='best')
plt.show()

### Projection Matrix Preparation

In [ ]:
# pairs of absolute eigenvalues and their respective eigenvectors
eigen_pairs = [(np.abs(eigen_vals[i]), eigen_vecs[:, i]) for i in range(len(eigen_vals))]
print('Eigenpairs:', eigen_pairs)

# sort the eigenpairs according to the absolute eigenvalues, from the largest to the smallest
eigen_pairs.sort(key=lambda k: k[0], reverse=True)
print('Sorted eigenpairs:', eigen_pairs)

In [ ]:
# W projection matrix computation
W = np.array([list(eigen_pairs[i][1]) for i in range(len(eigen_pairs))])
print('W projection shape:', W.shape)
print('W projection matrix:\n', W)

## 5 - Data Projection

In [ ]:
# 2D data projection
import matplotlib.pyplot as plt

print('X data shape:', X.shape)
print('W projection shape:', W.shape)

# matrix multiplication
pca_2d_X = norm_X @ W

# projected data plot
plt.scatter(pca_2d_X[:, 0], pca_2d_X[:, 1])
plt.show()

In [ ]:
# 1D data projection
import matplotlib.pyplot as plt


# matrix multiplication
pca_1d_X = norm_X @ W[:, 0]

# projected data plot
plt.scatter(pca_1d_X, np.zeros((len(pca_1d_X))))
plt.show()

## 6 - Scikit Implementation

In [ ]:
# same thing as before, but using scikit learn, 2D
from sklearn.decomposition import PCA
pca = PCA(n_components=2)

pca_2d_X = pca.fit_transform(norm_X) # it doesn't do standardization!
plt.scatter(pca_2d_X[:, 0], pca_2d_X[:, 1])
plt.show()

In [ ]:
# same thing as before, but using scikit learn, 1D
from sklearn.decomposition import PCA
pca = PCA(n_components=1)

pca_1d_X = pca.fit_transform(norm_X) # it doesn't do standardization!
plt.scatter(pca_1d_X, np.zeros((len(pca_1d_X))))
plt.show()

In [ ]:
# scikit allows us to choose the dimensions based on
# a given desired variance percentage
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95) # keep the dimensions that ensure 95% of variance

pca_95_X = pca.fit_transform(norm_X) # it doesn't do standardization!
plt.scatter(pca_95_X[:, 0], pca_95_X[:, 1])
plt.show()

## Activity

Download the "Wine Dataset" (see the code cell below) and execute data partition and preprocessing on it.   

Split the data into train and test partitions, with the test set containing 20% of the data.

Reduce the dimensionality of both train and test partitions making sure to keep at least 80% of the original data variance.

Plot the train and test datasets together on a 2D graph.

Make sure not to mix train and test partitions.   
**Do not contaminate the data!**

In [ ]:
# download the wine dataset
!curl -O https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data

# loads the wine dataset into memory
df_wine = pd.read_csv('/content/wine.data')

# adds headers to the dataset according to documentation
df_wine.columns = [
    'label', 'alcohol', 'malic acid', 'ash', 'alcalinity', 'magnesium',
    'phenols', 'flavanoids', 'nonflavanoid phenols', 'proanthocyanins',
    'color intensity', 'hue', 'protein concentration', 'proline']

# prints info
print('Data shape:', df_wine.shape)
print('Labels, Label count:', np.unique(df_wine['label'], return_counts=True))
print()

# first 10 samples
df_wine.head(10)